# Sieve Filter: Removing Small Raster Clumps

Classification outputs often contain salt-and-pepper noise: tiny clumps of 1-3 pixels that don't represent real features. The `sieve` function removes these by replacing connected regions smaller than a given threshold with the value of their largest spatial neighbor.

This is the xarray-spatial equivalent of GDAL's `gdal_sieve.py`, and it pairs naturally with classification functions like `natural_breaks()` or `reclassify()` and with `polygonize()` for cleaning results before vectorization.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from xrspatial.sieve import sieve
from xrspatial.classify import natural_breaks

## Generate a Noisy Classified Raster

We'll create a synthetic classified raster with three land-cover classes and scatter some salt-and-pepper noise across it.

In [ ]:
np.random.seed(42)
rows, cols = 80, 100

# Build a base classification with three broad zones
base = np.ones((rows, cols), dtype=np.float64)
base[:, 40:70] = 2.0
base[30:60, :] = 3.0
base[30:60, 40:70] = 2.0

# Add salt-and-pepper noise: randomly flip ~8% of pixels
noise_mask = np.random.random((rows, cols)) < 0.08
noise_vals = np.random.choice([1.0, 2.0, 3.0], size=(rows, cols))
noisy = base.copy()
noisy[noise_mask] = noise_vals[noise_mask]

# Sprinkle some NaN (nodata) pixels
noisy[0:3, 0:3] = np.nan
noisy[77:, 97:] = np.nan

raster = xr.DataArray(noisy, dims=['y', 'x'], name='landcover')
print(f'Raster shape: {raster.shape}')
print(f'Unique values (excl. NaN): {np.unique(raster.values[~np.isnan(raster.values)])}')

In [ ]:
cmap = ListedColormap(['#2ecc71', '#3498db', '#e74c3c'])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(raster.values, cmap=cmap, vmin=0.5, vmax=3.5, interpolation='nearest')
ax.set_title('Noisy classified raster')
cbar = fig.colorbar(im, ax=ax, ticks=[1, 2, 3])
cbar.ax.set_yticklabels(['Class 1', 'Class 2', 'Class 3'])
plt.tight_layout()
plt.show()

## Basic Sieve: Remove Single-Pixel Noise

The simplest use case: set a threshold so isolated pixels are absorbed by their surroundings.

In [ ]:
sieved = sieve(raster, threshold=4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in zip(axes, [raster, sieved], ['Before sieve', 'After sieve (threshold=4)']):
    im = ax.imshow(data.values, cmap=cmap, vmin=0.5, vmax=3.5, interpolation='nearest')
    ax.set_title(title)
fig.colorbar(im, ax=axes, ticks=[1, 2, 3], shrink=0.8)
plt.tight_layout()
plt.show()

## Connectivity: 4 vs 8

With 4-connectivity (rook), only pixels sharing an edge are considered connected. With 8-connectivity (queen), diagonally adjacent pixels also form part of the same region. This affects which clumps are identified as "small."

In [ ]:
sieved_4 = sieve(raster, threshold=6, neighborhood=4)
sieved_8 = sieve(raster, threshold=6, neighborhood=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, data, title in zip(
    axes,
    [raster, sieved_4, sieved_8],
    ['Original', '4-connectivity (threshold=6)', '8-connectivity (threshold=6)'],
):
    im = ax.imshow(data.values, cmap=cmap, vmin=0.5, vmax=3.5, interpolation='nearest')
    ax.set_title(title)
fig.colorbar(im, ax=axes, ticks=[1, 2, 3], shrink=0.8)
plt.tight_layout()
plt.show()

## Selective Sieving with `skip_values`

Sometimes certain class values should never be removed, even if their regions are small. Use `skip_values` to protect specific categories from merging while still allowing other small regions to be cleaned up.

In [ ]:
# Protect class 3 from sieving
sieved_skip = sieve(raster, threshold=10, skip_values=[3.0])
sieved_noskip = sieve(raster, threshold=10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, data, title in zip(
    axes,
    [raster, sieved_noskip, sieved_skip],
    ['Original', 'threshold=10 (no skip)', 'threshold=10 (skip class 3)'],
):
    im = ax.imshow(data.values, cmap=cmap, vmin=0.5, vmax=3.5, interpolation='nearest')
    ax.set_title(title)
fig.colorbar(im, ax=axes, ticks=[1, 2, 3], shrink=0.8)
plt.tight_layout()
plt.show()

## Practical Example: Clean Up a Classification

Generate a continuous surface, classify it with `natural_breaks`, and then sieve the result to remove small artifacts.

In [ ]:
# Create a smooth surface with some high-frequency variation
y = np.linspace(0, 4 * np.pi, rows)
x = np.linspace(0, 4 * np.pi, cols)
Y, X = np.meshgrid(y, x, indexing='ij')
surface = np.sin(Y) * np.cos(X) + 0.4 * np.random.randn(rows, cols)

surface_da = xr.DataArray(surface, dims=['y', 'x'])
classified = natural_breaks(surface_da, k=5)

# Sieve the classification
cleaned = sieve(classified, threshold=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(surface, cmap='terrain', interpolation='nearest')
axes[0].set_title('Continuous surface')
axes[1].imshow(classified.values, cmap='tab10', interpolation='nearest')
axes[1].set_title('natural_breaks (k=5)')
axes[2].imshow(cleaned.values, cmap='tab10', interpolation='nearest')
axes[2].set_title('After sieve (threshold=8)')
plt.tight_layout()
plt.show()

## Threshold Selection

The right threshold depends on pixel resolution and the minimum feature size you care about. Here's a comparison across threshold values.

In [ ]:
thresholds = [2, 5, 15, 50]
fig, axes = plt.subplots(1, len(thresholds), figsize=(5 * len(thresholds), 5))

for ax, t in zip(axes, thresholds):
    result = sieve(classified, threshold=t)
    ax.imshow(result.values, cmap='tab10', interpolation='nearest')
    ax.set_title(f'threshold={t}')

plt.suptitle('Effect of sieve threshold on classified raster', y=1.02)
plt.tight_layout()
plt.show()